# ternary_operator
Python’s ternary operator is a compact way to choose between two values. It can improve clarity when the decision is simple, but it becomes a maintenance problem when engineers try to compress too much business logic into one line.

## 1. Problem First
Why does this matter in real systems?
- Config resolution often picks between a provided value and a fallback.
- API responses may map a simple state into a label or action.
- One-line control flow can either improve readability or hide bugs in review.

In [1]:
status_code = 503
action = "retry" if status_code >= 500 else "process"
print(action)

retry


## 2. Minimal Concept
Core syntax:
- `value_if_true if condition else value_if_false`
- It returns a value, unlike a multi-line `if` block.
- It is best for short, obvious two-way decisions.

## 3. Mental Model
How Python actually behaves
A ternary expression is an expression, not a statement. Python evaluates the condition, then only one of the two result expressions. That means it can be used inline during assignment, return, or argument passing, but that convenience should not outrun readability.

In [2]:
latency_ms = 120
label = "slow" if latency_ms > 100 else "healthy"
print(label)

slow


## 4. Internal Mechanics
A ternary compiles into conditional branching just like an `if` statement, but it produces a value inline. Only the chosen branch expression is evaluated.

In [8]:
import dis

def classify(latency_ms):
    return "slow" if latency_ms > 100 else "healthy"

dis.dis(classify)
print(classify(50))

  3           0 RESUME                   0

  4           2 LOAD_FAST                0 (latency_ms)
              4 LOAD_CONST               1 (100)
              6 COMPARE_OP              68 (>)
             10 POP_JUMP_IF_FALSE        2 (to 16)
             12 LOAD_CONST               2 ('slow')
             14 RETURN_VALUE
        >>   16 LOAD_CONST               3 ('healthy')
             18 RETURN_VALUE
healthy


## 5. Edge Cases
Where it breaks:
- Nested ternaries become hard to read fast.
- Side effects inside the branches make reasoning harder.
- Using a ternary for multi-step business rules usually hides intent instead of clarifying it.

In [4]:
score = 81
grade = "A" if score >= 90 else "B" if score >= 80 else "C"
print(grade)

B


## 6. Performance Thinking
- Runtime cost is similar to a simple `if`/`else`.
- The real tradeoff is readability, not speed.
- Compact expressions reduce line count but can increase review cost if they hide branching logic.

## 7. Real Use Case
A CLI tool may pick a default output format based on whether structured mode is enabled.

In [5]:
structured_output = True
format_name = "json" if structured_output else "text"
print(format_name)

json


## 8. Anti-Pattern
What beginners do wrong:
- Use ternaries to look clever instead of to make intent clearer.
- Chain multiple ternaries where a normal `if` ladder would be easier to read.
- Hide important decisions inside long return statements.

In [6]:
cpu = 92
state = "critical" if cpu > 90 else "busy" if cpu > 70 else "healthy"
print(state)

if cpu > 90:
    clearer_state = "critical"
elif cpu > 70:
    clearer_state = "busy"
else:
    clearer_state = "healthy"
print(clearer_state)

critical
critical


## 9. Interview Signals
Questions FAANG asks:
- When is a ternary clearer than an `if` statement?
- Why are nested ternaries usually discouraged?
- Does a ternary evaluate both branches?
- How would you decide between compactness and readability in production code?

## 10. Exercise (Non-trivial)
You are reviewing a config loader that uses ternaries everywhere. Refactor the places where ternaries improve clarity, and replace the ones that obscure branching rules. Explain your cutoff for when a one-line conditional stops being maintainable.

In [7]:
def choose_region(cli_region, env_region):
    return cli_region if cli_region else env_region if env_region else "us-east-1"

# Task:
# 1. Decide whether this ternary chain is acceptable.
# 2. Refactor it if readability suffers.
# 3. Explain how falsey values could make this logic risky.